# 2 — Baseline Models
**Author:** Sai Srinivas Uppara

Trains and evaluates:
1. **BiLSTM + Attention** (text-only baseline)
2. **Gated Fusion** (multimodal baseline, 4 modalities)

Also loads the pretrained Gated Fusion and Cross-Attention weights from the teammate's training.

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.join(os.pardir, "src"))

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from config import (
    EMBEDDINGS_DIR, MODELS_DIR, RESULTS_DIR, PRETRAINED_DIR,
    TEXT_DIM, IMAGE_DIM, AUDIO_DIM, VIDEO_DIM, HIDDEN_DIM,
    NUM_CLASSES, DROPOUT, BATCH_SIZE, LEARNING_RATE, WEIGHT_DECAY,
    NUM_EPOCHS, PATIENCE, RANDOM_SEED,
)
from data_utils import load_embeddings, create_splits, make_dataloader
from models import BiLSTMTextOnly, GatedFusionModel, CrossAttentionModel
from train_utils import train_one_epoch, evaluate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
text, image, audio, video, labels = load_embeddings(EMBEDDINGS_DIR)
train_split, val_split, test_split = create_splits(
    text, image, audio, video, labels, seed=RANDOM_SEED
)

train_loader = make_dataloader(train_split, BATCH_SIZE, shuffle=True)
val_loader   = make_dataloader(val_split, BATCH_SIZE, shuffle=False)
test_loader  = make_dataloader(test_split, BATCH_SIZE, shuffle=False)

print(f"Train: {len(train_split[-1]):,}  Val: {len(val_split[-1]):,}  Test: {len(test_split[-1]):,}")

## Training helper

In [ ]:
def train_model(model, name, epochs=NUM_EPOCHS, patience=PATIENCE):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

    history = {"train_loss": [], "val_loss": [], "val_f1": []}
    best_f1, wait = 0.0, 0
    save_path = os.path.join(MODELS_DIR, f"{name}.pt")

    for ep in range(1, epochs + 1):
        tl, ta = train_one_epoch(model, train_loader, optimizer, criterion, device)
        vl, vm, _, _, _ = evaluate(model, val_loader, criterion, device)
        scheduler.step(vl)
        history["train_loss"].append(tl)
        history["val_loss"].append(vl)
        history["val_f1"].append(vm["f1"])
        print(f"  [{name}] Ep {ep:02d}  TrL={tl:.4f}  VL={vl:.4f}  VF1={vm['f1']:.4f}")
        if vm["f1"] > best_f1:
            best_f1 = vm["f1"]
            torch.save(model.state_dict(), save_path)
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"  Early stop at {ep}")
                break

    model.load_state_dict(torch.load(save_path, map_location=device, weights_only=True))
    _, test_met, _, _, _ = evaluate(model, test_loader, criterion, device)
    print(f"  [{name}] Test: {test_met}")

    with open(os.path.join(RESULTS_DIR, f"{name}_metrics.json"), "w") as f:
        json.dump({"test": test_met, "history": history}, f, indent=2)
    return model, test_met, history

## 2.1 BiLSTM + Attention (text-only)

In [ ]:
bilstm = BiLSTMTextOnly(text_dim=TEXT_DIM, hidden=HIDDEN_DIM,
                       num_classes=NUM_CLASSES, dropout=DROPOUT)
print(f"Params: {sum(p.numel() for p in bilstm.parameters()):,}")
bilstm_model, bilstm_met, bilstm_hist = train_model(bilstm, "baseline_bilstm")

## 2.2 Gated Fusion (multimodal) — train from scratch

In [ ]:
gf = GatedFusionModel(text_dim=TEXT_DIM, image_dim=IMAGE_DIM,
                     audio_dim=AUDIO_DIM, video_dim=VIDEO_DIM,
                     hidden=HIDDEN_DIM, num_classes=NUM_CLASSES, dropout=DROPOUT)
print(f"Params: {sum(p.numel() for p in gf.parameters()):,}")
gf_model, gf_met, gf_hist = train_model(gf, "baseline_gated_fusion")

## 2.3 Evaluate pretrained models

These were trained by the teammate and saved as `.pth` files.

In [ ]:
criterion = nn.CrossEntropyLoss()

# Pretrained Cross-Attention
ca = CrossAttentionModel(text_dim=TEXT_DIM, image_dim=IMAGE_DIM,
                         audio_dim=AUDIO_DIM, video_dim=VIDEO_DIM,
                         hidden=HIDDEN_DIM).to(device)
ca.load_state_dict(torch.load(
    os.path.join(PRETRAINED_DIR, "cross_attention_model.pth"),
    map_location=device, weights_only=True))
_, ca_met, _, _, _ = evaluate(ca, test_loader, criterion, device)
print(f"Pretrained Cross-Attention: {ca_met}")

with open(os.path.join(RESULTS_DIR, "pretrained_cross_attention_metrics.json"), "w") as f:
    json.dump({"test": ca_met}, f, indent=2)

# Pretrained Gated Fusion
gf_pre = GatedFusionModel(text_dim=TEXT_DIM, image_dim=IMAGE_DIM,
                          audio_dim=AUDIO_DIM, video_dim=VIDEO_DIM,
                          hidden=HIDDEN_DIM).to(device)
gf_pre.load_state_dict(torch.load(
    os.path.join(PRETRAINED_DIR, "gated_fusion_model.pth"),
    map_location=device, weights_only=True))
_, gf_pre_met, _, _, _ = evaluate(gf_pre, test_loader, criterion, device)
print(f"Pretrained Gated Fusion:    {gf_pre_met}")

with open(os.path.join(RESULTS_DIR, "pretrained_gated_fusion_metrics.json"), "w") as f:
    json.dump({"test": gf_pre_met}, f, indent=2)

## 2.4 Baseline comparison

In [ ]:
import pandas as pd

rows = [
    {"Model": "BiLSTM (text-only)", **bilstm_met},
    {"Model": "Gated Fusion (trained)", **gf_met},
    {"Model": "Cross-Attention (pretrained)", **ca_met},
    {"Model": "Gated Fusion (pretrained)", **gf_pre_met},
]
df = pd.DataFrame(rows).set_index("Model")
print(df.to_string())

df.plot(kind="bar", figsize=(12, 5), ylim=(0.5, 1.0), rot=15)
plt.title("Baseline Comparison")
plt.ylabel("Score")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()